# Teks — Fine-tune transformer multilabel (BCE) k-fold + OOF + ambang per label
**Kapan dipakai:** satu teks bisa punya BANYAK label sekaligus (aspek ulasan, jenis ujaran
kebencian, tag topik), target berupa beberapa kolom 0/1. `multilabel` (TF-IDF) = jalur cepat;
notebook ini jalur skor tinggi dengan GPU.

**Beda dengan klasifikasi biasa:** keluaran = satu logit per label, loss `BCEWithLogits`
(alt `pos_weight` untuk label langka), sigmoid (bukan softmax), dan AMBANG tiap label dicari dari
OOF (0.5 jarang optimal untuk label langka). Aturan "minimal satu label" opsional.

**Alur:** Konfigurasi → baca → k-fold (`KFold`) → latih tiap fold → OOF → ambang per label →
simpan OOF → submission kolom 0/1 → validator.

**Saat lomba yang diubah:** sel Konfigurasi (`LABEL_COLS`). Waktu mepet: `FOLDS_TO_RUN = [0]`.

**Data demo:** CASA (`multilabel`, 6 aspek ulasan mobil, 810 baris). Bentuk lain:
`ujaran_kebencian` (12 label, 10 rb baris).

In [ ]:
import html
import os
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/multilabel"         # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/casa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/casa_test.csv"                      # GANTI saat lomba
ANSWER_CSV = f"{DATA_FOLDER}/casa_test_labels.csv"             # HAPUS saat lomba (kunci demo)
# DATA_FOLDER = "../../../data_latihan/text/ujaran_kebencian"  # bentuk lain: 12 label
# TRAIN_CSV = f"{DATA_FOLDER}/train.csv"
# TEST_CSV = f"{DATA_FOLDER}/test.csv"
# ANSWER_CSV = f"{DATA_FOLDER}/test_labels.csv"
OUTPUT_FOLDER = "../../../outputs/text/finetune_multilabel"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ID_COL = "id"                    # GANTI saat lomba
TEXT_COL = "text"                # GANTI saat lomba
LABEL_COLS = None                # None = semua kolom selain ID_COL dan TEXT_COL
# LABEL_COLS = ["fuel", "machine", "others", "part", "price", "service"]
N_FOLDS = 5
SPLIT_SEED = 42
NOTEBOOK = "finetune_multilabel"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"

In [ ]:
PRETRAINED = "../../../pretrained/indobert"           # GANTI saat lomba
# PRETRAINED = "../../../pretrained/indobertweet"     # medsos (ujaran kebencian): lebih cocok
# PRETRAINED = "../../../pretrained/xlmr_base"        # campur bahasa / daerah
MODEL_NAME = "ml_indobert"       # GANTI tiap variasi (nama file OOF)
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 5                       # data kecil butuh epoch lebih banyak
LEARNING_RATE = 3e-5
WARMUP = 0.1
POS_WEIGHT = False               # True: bobot label positif = negatif/positif (label langka)
MIN_ONE_LABEL = False            # True: tiap teks minimal 1 label (label skor tertinggi)
FOLDS_TO_RUN = None              # None = semua fold. Waktu mepet: [0] (OOF tidak disimpan)
SEED = 42

## Metrik
Default F1 makro antarlabel (rata-rata F1 tiap kolom). Alternatif di baris komentar.

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro", zero_division=0)
    # return f1_score(truth, predicted, average="micro", zero_division=0)
    # return f1_score(truth, predicted, average="samples", zero_division=0)


print("metrik:", METRIC_NAME, "| device:", DEVICE)

## Baca data

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
if LABEL_COLS is None:
    LABEL_COLS = []
    for column in train.columns:
        if column not in [ID_COL, TEXT_COL]:
            LABEL_COLS.append(column)


def clean_basic(text):
    return " ".join(html.unescape(text).split())


train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
train_text = train_text.apply(clean_basic)
test_text = test_text.apply(clean_basic)
label_matrix = train[LABEL_COLS].to_numpy().astype("float32")
print("train:", train.shape, "| test:", test.shape, "| label:", LABEL_COLS)
print("positif per label:", dict(zip(LABEL_COLS, label_matrix.sum(axis=0).astype(int))))
per_text = pd.Series(label_matrix.sum(axis=1)).value_counts()
print("label per teks:", per_text.sort_index().to_dict())

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
for column in LABEL_COLS:
    demo_sample[column] = 0
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())

## Tokenisasi dan split
`KFold` acak seed 42 (format OOF bersama untuk multilabel). Iterative stratification lebih
rapi untuk label langka, tapi paketnya tidak terpasang.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)


def encode_rows(texts):
    encoded = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
    rows = []
    for number in range(len(encoded["input_ids"])):
        row = {}
        for key in encoded.keys():
            row[key] = encoded[key][number]
        rows.append(row)
    return rows


train_rows = encode_rows(train_text)
test_rows = encode_rows(test_text)
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED).split(train_rows))
lengths = []
for row in train_rows[:2000]:
    lengths.append(len(row["input_ids"]))
print("token p95:", np.percentile(lengths, 95), "| fold:", len(folds))

## Model, latih, prediksi

In [ ]:
def pick(rows, chosen):
    picked = []
    for number in chosen:
        picked.append(rows[number])
    return picked


def forward_logits(model, rows):
    batch = tokenizer.pad(rows, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        return model(**batch).logits.float()


def predict_proba(model, rows):
    model.eval()
    proba = np.zeros((len(rows), len(LABEL_COLS)), dtype="float32")
    with torch.no_grad():
        for start in range(0, len(rows), BATCH_SIZE * 4):
            chosen = np.arange(start, min(start + BATCH_SIZE * 4, len(rows)))
            logits = forward_logits(model, pick(rows, chosen))
            proba[chosen] = torch.sigmoid(logits.cpu()).numpy()
    return proba


print("predict_proba siap")

In [ ]:
def build_run(fit_index):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=len(LABEL_COLS), ignore_mismatched_sizes=True,
        local_files_only=True, dtype=torch.float32)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = max(1, int(np.ceil(len(fit_index) / BATCH_SIZE))) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
    pos_weight = None
    if POS_WEIGHT:
        positive = np.maximum(label_matrix[fit_index].sum(axis=0), 1)
        pos_weight = torch.tensor((len(fit_index) - positive) / positive).to(DEVICE)
    return {"model": model, "optimizer": optimizer, "scheduler": scheduler,
            "pos_weight": pos_weight, "rng": np.random.default_rng(SEED)}


print("build_run siap | POS_WEIGHT", POS_WEIGHT)

In [ ]:
def train_epoch(run, fit_index):
    model = run["model"]
    model.train()
    order = run["rng"].permutation(fit_index)
    for start in range(0, len(order), BATCH_SIZE):
        chosen = order[start:start + BATCH_SIZE]
        logits = forward_logits(model, pick(train_rows, chosen))
        targets = torch.tensor(label_matrix[chosen]).to(DEVICE)
        loss = binary_cross_entropy_with_logits(logits, targets, pos_weight=run["pos_weight"])
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        run["optimizer"].step()
        run["scheduler"].step()
        run["optimizer"].zero_grad()


def snapshot(model):
    state = {}
    for key, tensor in model.state_dict().items():
        moved = tensor.detach().cpu()
        state[key] = moved.clone()
    return state


print("train_epoch siap")

In [ ]:
def train_fold(fit_index, val_index):
    run = build_run(fit_index)
    val_rows = pick(train_rows, val_index)
    best, state = -1.0, None
    for epoch in range(EPOCHS):
        began = time.time()
        train_epoch(run, fit_index)
        value = score(label_matrix[val_index], predict_proba(run["model"], val_rows) >= 0.5)
        if value > best:
            best, state = value, snapshot(run["model"])
        print(f"  epoch {epoch + 1}: {METRIC_NAME}@0.5 {value:.4f} | {time.time() - began:.0f} dtk")
    run["model"].load_state_dict(state)
    val_proba = predict_proba(run["model"], val_rows)
    test_part = predict_proba(run["model"], test_rows)
    del run
    torch.cuda.empty_cache()
    return val_proba, test_part


print("train_fold siap")

## Latih k-fold

In [ ]:
folds_to_run = FOLDS_TO_RUN
if folds_to_run is None:
    folds_to_run = list(range(len(folds)))
oof = np.zeros(label_matrix.shape, dtype="float32")
filled = np.zeros(len(train), dtype=bool)
test_proba = np.zeros((len(test), len(LABEL_COLS)), dtype="float32")
began = time.time()
for number in folds_to_run:
    fit_index, val_index = folds[number]
    val_proba, test_part = train_fold(fit_index, val_index)
    oof[val_index] = val_proba
    filled[val_index] = True
    test_proba += test_part / len(folds_to_run)
    fold_value = score(label_matrix[val_index], val_proba >= 0.5)
    print(f"fold {number}: {METRIC_NAME}@0.5 {fold_value:.4f}")
total_seconds = round(time.time() - began, 1)
print(f"total {total_seconds / 60:.1f} menit")

## Ambang per label dari OOF
Untuk tiap label dicari ambang 0.05..0.95 yang memberi F1 OOF tertinggi. Ambang dari OOF tidak
bocor ke test. Label dengan positif sangat sedikit: ambang bisa melompat, periksa tabelnya.

In [ ]:
grid = np.arange(0.05, 0.96, 0.05)
thresholds = np.full(len(LABEL_COLS), 0.5)
for column in range(len(LABEL_COLS)):
    best_f1 = -1.0
    for cut in grid:
        value = f1_score(label_matrix[filled, column], oof[filled, column] >= cut, zero_division=0)
        if value > best_f1:
            best_f1, thresholds[column] = value, cut


def to_labels(proba):
    labels = (proba >= thresholds).astype(int)
    if MIN_ONE_LABEL:
        empty = labels.sum(axis=1) == 0
        labels[empty, proba[empty].argmax(axis=1)] = 1
    return labels


oof_score = score(label_matrix[filled], to_labels(oof[filled]))
threshold_table = pd.Series(thresholds, index=LABEL_COLS)
print(threshold_table.round(2).to_dict())
plain_score = score(label_matrix[filled], oof[filled] >= 0.5)
print(f"{METRIC_NAME} OOF @0.5 {plain_score:.4f} | ambang per label {oof_score:.4f}")

## Simpan OOF dan catat percobaan
Disimpan probabilitas sigmoid (n_train × n_label), bukan 0/1.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
note = f"{os.path.basename(PRETRAINED)} | epoch {EPOCHS} lr {LEARNING_RATE} len {MAX_LENGTH}"
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK, "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(3).to_string())

## Submission
Satu kolom 0/1 per label, urutan kolom dan id mengikuti `sample_submission`.

In [ ]:
test_labels = pd.DataFrame(to_labels(test_proba), columns=LABEL_COLS)
test_labels.index = test[ID_COL].to_numpy()

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
for column in sample.columns:
    if column != ID_COL:
        submission[column] = sample[ID_COL].map(test_labels[column]).to_numpy()
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
labels_per_text = submission[LABEL_COLS].sum(axis=1)
print("rata-rata label per teks test:", round(labels_per_text.mean(), 2))
submission.head()

## Cek submission terhadap sample_submission
Versi ringkas validator `simpan_muat` #15–#16 untuk kolom 0/1. Harus `MASALAH: 0`.

In [ ]:
written = pd.read_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}")
problems = []
if written.columns.tolist() != sample.columns.tolist():
    problems.append("kolom / urutan kolom beda")
if written[ID_COL].tolist() != sample[ID_COL].tolist():
    problems.append("id / urutan id beda dari sample")
for column in LABEL_COLS:
    if written[column].isna().any():
        problems.append(f"{column}: ada nilai kosong")
    if not set(written[column].unique()) <= {0, 1}:
        problems.append(f"{column}: nilai selain 0/1")
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
answers.index = answers[ID_COL].to_numpy()
truth = answers.loc[submission[ID_COL], LABEL_COLS].to_numpy()
print(f"{METRIC_NAME} di test (kunci): {score(truth, submission[LABEL_COLS].to_numpy()):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")